<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта: Реализация класса Invoice</h2>

----

### Вариант задания №10


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс Invoice в C#, который будет представлять информацию о фактурах за поставленные товары или оказанные услуги. На основе этого класса разработать 2-3 производных класса, демонстрирующих принципы наследования и полиморфизма. В каждом из классов должны быть реализованы новые атрибуты и методы, а также переопределены некоторые методы базового класса для
демонстрации полиморфизма.
Требования к базовому классу Invoice:

• Атрибуты: Номер фактуры (InvoiceNumber), Дата выдачи (IssueDate), Общая
сумма (TotalAmount).

• Методы:

o CalculateTotal(): метод для расчета общей суммы по фактуре.

o AddLine(LineItem lineItem): метод для добавления позиции в фактуру.

o RemoveLine(LineItem lineItem): метод для удаления позиции из фактуры.

Требования к производным классам:

1. ТоварнаяФактура (GoodsInvoice): Должна содержать дополнительные атрибуты, такие как Дата поставки (SupplyDate). Метод AddLine() должен быть переопределен для добавления информации о дате поставки товара при добавлении позиции.

2. УслуговаяФактура (ServiceInvoice): Должна содержать дополнительные атрибуты, такие как Дата оказания услуги (ServiceDate). Метод RemoveLine() должен быть переопределен для добавления информации о причине аннулирования услуги при удалении позиции.

3. КомбинированнаяФактура (CombinedInvoice) (если требуется третий класс): Должна содержать дополнительные атрибуты, такие как Наличие возврата (ReturnAllowed). Метод CalculateTotal() должен быть переопределен для учета возможного возврата товара или услуги при расчете общей суммы.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) исользуйтие в проекте коллекции, делегаты, события.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [ ]:
using System;
using System.Collections.Generic;
using System.Globalization;

// ============================================================================
// Вспомогательные классы
// ============================================================================
public class LinItem // Класс позиции в фактуре
{
    public string ItemName { get; set; }      // Наименование товара/услуги
    public double Price { get; set; }        // Цена
    public int Quantity { get; set; }        // Количество

    public override string ToString()
    {
        return $"{ItemName}: {Quantity} x {Price:C} = {(Price * Quantity):C}";
    }
}

// ============================================================================
// Базовый класс Invoice
// ============================================================================
// Объявляем делегат для обработки изменения суммы (Требование: Делегаты)
public delegate void TotalAmountChangedDelegate(Invoice source, string message);

public class Invoice
{
    // Атрибуты
    public string InvoiceNumber { get; set; }              // Номер фактуры
    public DateTime IssueDate { get; set; }                // Дата выдачи
    public double TotalAmount { get; set; }                // Общая сумма
    
    // Дополнительный атрибут (Требование: добавить атрибуты)
    public string CustomerId { get; set; }                 // ID клиента
    private List<LinItem> _lineItems;                     // Коллекция позиций (Требование: Коллекции)

    // Событие (Требование: События)
    public event TotalAmountChangedDelegate TotalChanged;

    public Invoice(string number, DateTime date, string customerId)
    {
        InvoiceNumber = number;
        IssueDate = date;
        CustomerId = customerId;
        _lineItems = new List<LinItem>();                   // Инициализация коллекции
        TotalAmount = 0;
    }

    // Методы
    public virtual void CalculateTotal()
    {
        TotalAmount = 0;
        foreach (var item in _lineItems)
        {
            TotalAmount += item.Price * item.Quantity;
        }
        
        // Вызов события при изменении суммы
        InvokeTotalChanged($"Сумма пересчитана: {TotalAmount:C}");
    }

    public virtual void AddLine(LinItem linItem)
    {
        _lineItems.Add(linItem);
        CalculateTotal();
    }

    public virtual void RemoveLine(LinItem linItem)
    {
        if (_lineItems.Contains(linItem))
        {
            _lineItems.Remove(linItem);
            CalculateTotal();
        }
    }

    // Виртуальный метод для демонстрации полиморфизма
    public virtual void DisplayDetails()
    {
        Console.WriteLine($"\n--- Фактура #{InvoiceNumber} ---");
        Console.WriteLine($"Дата: {IssueDate:dd.MM.yyyy} | Клиент: {CustomerId}");
        Console.WriteLine("Позиции:");
        foreach (var item in _lineItems)
        {
            Console.WriteLine($"  {item}");
        }
        Console.WriteLine($"ИТОГО: {TotalAmount:C}");
    }

    // Защита вызова события
    protected virtual void InvokeTotalChanged(string message)
    {
        // Если подписчик есть, вызываем его
        TotalChanged?.Invoke(this, message);
    }
}

// ============================================================================
// Производный класс: ТоварнаяФактура (GoodsInvoice)
// ============================================================================
public class GoodsInvoice : Invoice
{
    // Дополнительный атрибут
    public DateTime SupplyDate { get; set; }               // Дата поставки

    public GoodsInvoice(string number, DateTime issueDate, string customerId, DateTime supplyDate)
        : base(number, issueDate, customerId)
    {
        SupplyDate = supplyDate;
    }

    // Переопределение метода AddLine (Требование)
    public override void AddLine(LinItem linItem)
    {
        // Логика добавления именно товарной позиции
        Console.WriteLine("[GoodsInvoice] Добавление товара...");
        base.AddLine(linItem);
    }

    // Переопределение метода DisplayDetails
    public override void DisplayDetails()
    {
        Console.WriteLine("\n--- ТОВАРНАЯ ФАКТУРА ---");
        base.DisplayDetails();
        Console.WriteLine($"Дата поставки: {SupplyDate:dd.MM.yyyy}");
    }
}

// ============================================================================
// Производный класс: УсловнаяФактура (ServiceInvoice)
// ============================================================================
public class ServiceInvoice : Invoice
{
    // Дополнительный атрибут
    public DateTime ServiceDate { get; set; }              // Дата оказания услуги
    
    // Для демонстрации RemoveLine (Требование)
    private string LastRemovalReason { get; set; } = string.Empty; 

    public ServiceInvoice(string number, DateTime issueDate, string customerId, DateTime serviceDate)
        : base(number, issueDate, customerId)
    {
        ServiceDate = serviceDate;
    }

    // Переопределение метода RemoveLine (Требование)
    public override void RemoveLine(LinItem linItem)
    {
        string reason = "Отмена услуги без указания причины"; // По умолчанию
        
        // Доп. логика: например, запрашиваем причину для вывода
        if (linItem.ItemName == "Консультация") 
            reason = "Консультация была отменена по графику";

        LastRemovalReason = reason;
        
        Console.WriteLine($"[ServiceInvoice] Удаление позиции с причиной: {reason}");
        base.RemoveLine(linItem);
    }

    // Переопределение метода DisplayDetails
    public override void DisplayDetails()
    {
        Console.WriteLine("\n--- УСЛОВНАЯ ФАКТУРА ---");
        base.DisplayDetails();
        Console.WriteLine($"Дата услуги: {ServiceDate:dd.MM.yyyy}");
    }
}

// ============================================================================
// Производный класс: ВозвратнаяФактура (ReturnInvoice - вариант 3)
// ============================================================================
public class ReturnInvoice : Invoice
{
    // Дополнительный атрибут (если требуется третий класс)
    public bool ReturnAllowed { get; set; }                // Возможность возврата
    public string ReturnComment { get; set; }              // Комментарий к возврату

    public ReturnInvoice(string number, DateTime issueDate, string customerId, bool allowed)
        : base(number, issueDate, customerId)
    {
        ReturnAllowed = allowed;
    }

    // Переопределение CalculateTotal (для учета отрицательных сумм или скидок)
    public override void CalculateTotal()
    {
        double sum = 0;
        foreach (var item in _lineItems)
        {
            if (ReturnAllowed)
                sum -= (item.Price * item.Quantity); // Пример учета как минусовой операции
            else
                sum += (item.Price * item.Quantity);
        }
        TotalAmount = sum;
        
        if(ReturnAllowed)
            InvokeTotalChanged($"Сформирован возврат на сумму {Math.Abs(TotalAmount):C}");
        else
            InvokeTotalChanged($"Пересчет завершил: {TotalAmount:C}");
    }

    public override void DisplayDetails()
    {
        Console.WriteLine("\n--- ФАКТУРА ВОЗВРАТА ---");
        base.DisplayDetails();
        Console.WriteLine($"Возврат разрешен: {ReturnAllowed}");
        if (!string.IsNullOrEmpty(ReturnComment))
            Console.WriteLine($"Комментарий: {ReturnComment}");
    }
}

// ============================================================================
// Программа (Main)
// ============================================================================
class Program
{
    static void Main()
    {
        // Подписчики событий (Использование делегатов и событий)
        // Статический метод для обработки изменений
        Invoice.TotalAmountChanged += HandleTotalChangeLog;
        // Lambda-выражение для мгновенной реакции
        Invoice.TotalAmountChanged += (source, msg) => Console.WriteLine($"[{DateTime.Now:HH:mm:ss}] Лямбда: {msg}");

        Console.WriteLine("=== Создание объектов ===");

        // 1. Создаем обычную товарную фактуру
        var goodsInv = new GoodsInvoice(
            "INV-001", 
            DateTime.Today, 
            "Client-A", 
            DateTime.Today.AddDays(5)
        );
        
        goodsInv.AddLine(new LinItem { ItemName = "Laptop XPS", Price = 1000, Quantity = 1 });
        goodsInv.AddLine(new LinItem { ItemName = "Mouse", Price = 50, Quantity = 2 });

        // 2. Создаем фактуру услуги
        var serviceInv = new ServiceInvoice(
            "SRV-001",
            DateTime.Today,
            "Client-B",
            DateTime.Today.AddDays(1)
        );
        
        serviceInv.AddLine(new LinItem { ItemName = "IT Consulting", Price = 200, Quantity = 1 });

        // 3. Создаем фактуру возврата
        var returnInv = new ReturnInvoice(
            "RET-001",
            DateTime.Today,
            "Client-C",
            true // ReturnAllowed = true
        );
        returnInv.ReturnComment = "Не подошел размер";
        returnInv.AddLine(new LinItem { ItemName = "Shoes Nike", Price = 150, Quantity = 1 });


        Console.WriteLine("\n=== Отображение информации (Полиморфизм) ===");
        goodsInv.DisplayDetails();
        serviceInv.DisplayDetails();
        returnInv.DisplayDetails();


        Console.WriteLine("\n=== Изменение данных (Триггеры событий) ===");
        
        // Попытка удалить позицию из услугной фактуры
        // Сначала находим её в списке (упрощенно создадим новый объект для поиска)
        LinItem consultingItem = new LinItem { ItemName = "IT Consulting", Price = 200, Quantity = 1 };
        serviceInv.RemoveLine(consultingItem);

        // Пересчет общей суммы вызовет событие
        // Мы уже знаем, что было пересчитано, но демонстрируем логику
        goodsInv.CalculateTotal(); 
        returnInv.CalculateTotal();

        Console.WriteLine("\n=== Очистка подписок (Опционально) ===");
        // Можно отписать статический обработчик, чтобы не дублировать логику в будущем
        // Invoice.TotalAmountChanged -= HandleTotalChangeLog;
    }

    // Статический метод-обработчик события (Используется как делегат)
    static void HandleTotalChangeLog(Invoice source, string message)
    {
        Console.WriteLine($">>> СИСТЕМНЫЙ ЛОГ: [ID={source.InvoiceNumber}] {message}");
    }
}